# FLAME 3: incendio

Preprocesamiento del experimento de incendio con [FLAME 3 – Nadir Thermal Plot Subset](https://www.kaggle.com/datasets/brycehopkins/flame-3-nadir-thermal-plot-subset)
(Hanna Hammock, Tall Timbers, Florida; quema prescrita del 2023-02-28). Un dron (Autel EVO II Dual 640T, cámara
XT709) a 122 m sobre el centro de cada plot toma una imagen térmica cada 5 s. Cada `IRX_XXXX_ref_geo.TIFF` es un
raster `float32` en °C absolutos, georreferenciado en EPSG:26916 (UTM 16N, ~0.15 m/px); las zonas que la cámara no vio
quedan en 0 °C (sin dato). Los GeoTIFF no traen fecha: se toma del JPG con el mismo número `IRX_XXXX`.

| Plot | JPG | GeoTIFF °C | Inicio → fin | Huecos > 10 s |
|---|---|---|---|---|
| plot 1 | 297 (`IRX_0528`–`0824`) | 296 (`IRX_0529`–`0824`) | 11:46:19 → 12:11:32 | 15 s y 26 s al inicio |
| plot 2 | 522 (`IRX_0825`–`1346`) | 498, en `part1` (`0826`–`1147`) y `part2` (`1171`–`1346`) | 12:41:27 → 13:32:29 | 460 s entre `1147` y `1171` |
| plot 3 | — | — | — | sin imágenes térmicas |

Flujo (cada paso lee la salida del anterior):

1. **Descarga y selección de frames** → `Dataset/FLAME3/seleccion2/<escenario>/`
2. **Alineación** → `geo_thermal_tiff_celsius_aligned/`
3. **Recorte cuadrado** → `geo_thermal_tiff_celsius_aligned_<lado>/`
4. **Reducción** a 100×100 → `geo_thermal_tiff_celsius_aligned_100/`
5. **Número de estados** (análisis: *k-means* y comparación visual de divisiones)
6. **Estados logarítmicos** (15) → `CA/FLAME3/seleccion2/<escenario>/`
7. **Unión de estados fríos** (12) → `CA/FLAME3/seleccion2/<escenario>/estados12/`
8. **Estados de quemado** (12 + 6) → `CA/FLAME3/seleccion2/BurnedStates/` (datos del experimento final)

Anexo: quitar la ignición (selección 1, experimentos anteriores).

In [ ]:
import csv
import json
import re
import shutil
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import rasterio
import scipy.io
from matplotlib import colormaps
from PIL import Image

TRAINING = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "main.py").exists())
SIMULATOR = TRAINING.parent                     # las paletas se guardan como GUI/colors/<nombre>.csv relativo a aquí
DATASET = TRAINING / "Dataset" / "FLAME3"
CA = TRAINING / "CA" / "FLAME3"


def to_rgb(frame: np.ndarray, vmin: float, vmax: float) -> np.ndarray:
    """Pseudocolor inferno con escala logarítmica fija; "sin dato" en negro (JPG para revisar a ojo)."""
    x = np.clip(np.log(np.maximum(frame, vmin) / vmin) / np.log(vmax / vmin), 0, 1)
    rgb = (colormaps["inferno"](x)[..., :3] * 255).astype(np.uint8)
    rgb[frame <= 0] = 0
    return rgb


def palette(k: int) -> np.ndarray:
    """Paleta turbo en k pasos, sin los extremos (casi negro y rojo muy oscuro)."""
    return (colormaps["turbo"](np.linspace(0.05, 0.95, k))[:, :3] * 255).astype(np.uint8)


def leer_tiff(carpeta: Path) -> tuple:
    paths = sorted(carpeta.glob("*.TIFF"))
    if not paths:
        sys.exit(f"No hay GeoTIFF en {carpeta}")
    return paths, np.stack([rasterio.open(p).read(1) for p in paths])


def guardar_evolucion(out: Path, evo: np.ndarray, celsius: np.ndarray, colores: np.ndarray, meta: dict, zoom=4, fps=10):
    """evo.npy / evo.mat (variable `evolution`), evo_celsius.npy, evo.json, evo.gif y evo_final.png."""
    out.mkdir(parents=True, exist_ok=True)
    np.save(out / "evo.npy", evo)
    np.save(out / "evo_celsius.npy", celsius)
    scipy.io.savemat(out / "evo.mat", {"evolution": evo})
    frames = [Image.fromarray(colores[st].repeat(zoom, 0).repeat(zoom, 1)) for st in evo]
    frames[0].save(out / "evo.gif", save_all=True, append_images=frames[1:], duration=int(1000 / fps), loop=0)
    frames[-1].save(out / "evo_final.png")
    (out / "evo.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))


def secuencias(evo: np.ndarray, carpeta: Path, step: int):
    """Secuencias de entrenamiento con paso de `step` frames: con paso k la NCA aprende estado(t) → estado(t + k). Para
    no tirar frames, la evolución se parte en k secuencias con desfase o = 0 … k−1 → entrenamiento/k<k>_o<o>.npy / .mat."""
    ent = carpeta / "entrenamiento"
    ent.mkdir(exist_ok=True)
    for o in range(step):
        s = evo[o::step]
        np.save(ent / f"k{step}_o{o}.npy", s)
        scipy.io.savemat(ent / f"k{step}_o{o}.mat", {"evolution": s})
    print(f"  {step} secuencias con paso {step} ({step * 5} s) → {ent}")


def leer_paleta(nombre: str) -> list:
    return [h.strip() for h in (SIMULATOR / nombre).read_text().split()]


def hex_a_rgb(colores: list) -> np.ndarray:
    return np.array([[int(h[i:i + 2], 16) for i in (1, 3, 5)] for h in colores], dtype=np.uint8)

## 1. Descarga y selección de frames

El dataset completo (35 GB) se baja con `kagglehub` a `~/.cache/kagglehub/`. `seleccionar` copia los GeoTIFF
`IRX_XXXX_ref_geo.TIFF` y los JPG `IRX_XXXX.jpg` con `first ≤ XXXX ≤ last` de `<plot>/duringburn/` (incluidas
subcarpetas como `part1`/`part2`) a `<salida>/geo_thermal_tiff_celsius/` y `<salida>/raw_thermal_jpg/`.

In [ ]:
def numero(p: Path) -> int:
    return int(re.search(r"IRX_(\d+)", p.name).group(1))


def seleccionar(plot, salida, first, last):
    origen, salida = Path(plot) / "duringburn", Path(salida)
    for sub, patron in (("geo_thermal_tiff_celsius", "IRX_*_ref_geo.TIFF"), ("raw_thermal_jpg", "IRX_*.jpg")):
        archivos = sorted((p for p in (origen / sub).rglob(patron) if first <= numero(p) <= last), key=numero)
        destino = salida / sub
        destino.mkdir(parents=True, exist_ok=True)
        for p in archivos:
            shutil.copy2(p, destino / p.name)
        print(f"{sub}: {len(archivos)} archivos ({archivos[0].name} – {archivos[-1].name}) → {destino}")

Selección 2, la del experimento final: `flame_plot_1` = plot 1 `IRX_0528`–`0654` (antes de que arranque el fuego),
`flame_plot_2` = plot 1 `0662`–`0824` y `flame_plot_3` = plot 2 `1189`–`1346`.

In [ ]:
import kagglehub

KAGGLE = Path(kagglehub.dataset_download("brycehopkins/flame-3-nadir-thermal-plot-subset")) / "Hanna Hammock"
ESCENARIOS = [DATASET / "seleccion2" / f"flame_plot_{i}" for i in (1, 2, 3)]
seleccionar(KAGGLE / "plot 1", ESCENARIOS[0], 528, 654)
seleccionar(KAGGLE / "plot 1", ESCENARIOS[1], 662, 824)
seleccionar(KAGGLE / "plot 2", ESCENARIOS[2], 1189, 1346)

## 2. Alineación

Los GeoTIFF ya comparten la misma malla, pero queda una deriva residual de hasta ~9 px a lo largo de la secuencia.

1. Mide la traslación subpíxel entre pares de frames (correlación de fase) a distancias 1, 2, 4, …, 128, usando solo la
   textura del suelo frío (< `cool` °C en ambos frames, con un margen de 12 px alrededor del fuego). Los pares a
   distancia 1, 2 y 4 se miden además con la textura de log(°C) en toda la imagen, con peso bajo, para que ningún frame
   quede sin medición cuando el fuego cubre casi todo.
2. Resuelve la posición de todos los frames a la vez por mínimos cuadrados robustos (Huber, c = 1 px), con el primer
   frame fijo. Medir contra varios frames evita que el error se acumule como al encadenar frames consecutivos (hasta
   5 px en plot 2); así la posición coincide a < 0.2 px con la medida directa contra el primer frame.
3. Desplaza cada frame a la posición del primero con interpolación bilineal; lo que queda fuera pasa a 0 °C.
4. Guarda los GeoTIFF alineados, un JPG por frame en `raw_thermal_jpg_aligned/` (misma escala logarítmica en toda la
   secuencia, para revisar a ojo) y `shifts.csv` (corrección de cada frame en px).

In [ ]:
def largest_rectangle(mask: np.ndarray) -> tuple:
    """Rectángulo alineado a los ejes de mayor área con todo True. Regresa (r0, r1, c0, c1)."""
    H, W = mask.shape
    heights = np.zeros(W, dtype=int)
    best_area, best = 0, (0, 0, 0, 0)
    for r in range(H):
        heights = np.where(mask[r], heights + 1, 0)
        stack = []  # (columna inicial, altura), alturas crecientes
        for c in range(W + 1):
            h = heights[c] if c < W else 0
            start = c
            while stack and stack[-1][1] >= h:
                s, sh = stack.pop()
                if sh * (c - s) > best_area:
                    best_area, best = sh * (c - s), (r - sh + 1, r + 1, s, c)
                start = s
            stack.append((start, h))
    return best


def highpass(x: np.ndarray, sigma: float = 4) -> np.ndarray:
    from scipy import ndimage

    return x - ndimage.gaussian_filter(x, sigma)


def measure_positions(stack: np.ndarray, cool: float, upsample: int,
                      lags=(1, 2, 4, 8, 16, 32, 64, 128), log_lags=(1, 2, 4),
                      log_weight=0.02, margin=12, min_overlap=0.05) -> tuple:
    """Posición (fila, col) de cada frame respecto al primero y residuo de cada par medido."""
    from scipy import ndimage
    from skimage.registration import phase_cross_correlation

    T = len(stack)
    window = np.outer(np.hanning(stack.shape[1]), np.hanning(stack.shape[2]))
    frames = stack.astype(np.float64)
    # suelo frío con margen alrededor del fuego y borde suavizado
    cold = np.stack([ndimage.gaussian_filter(ndimage.binary_erosion(f < cool, iterations=margin).astype(float), 3)
                     for f in frames])
    valid = np.stack([ndimage.binary_erosion(f > 0, iterations=margin) for f in frames])
    cold_tex = np.stack([highpass(np.minimum(f, cool)) for f in frames])
    log_tex = np.stack([highpass(np.log(np.maximum(f, 1))) for f in frames])

    pairs, meas, weights = [], [], []
    for lag in lags:
        for t in range(lag, T):
            m = cold[t - lag] * cold[t]
            if (m > 0.5).mean() < min_overlap:
                continue
            s, _, _ = phase_cross_correlation(cold_tex[t - lag] * m * window, cold_tex[t] * m * window,
                                              upsample_factor=upsample, normalization=None)
            pairs.append((t - lag, t)); meas.append(s); weights.append((m > 0.5).mean())
    for lag in log_lags:
        for t in range(lag, T):
            m = (valid[t - lag] & valid[t]) * window
            s, _, _ = phase_cross_correlation(log_tex[t - lag] * m, log_tex[t] * m,
                                              upsample_factor=upsample, normalization=None)
            pairs.append((t - lag, t)); meas.append(s); weights.append(log_weight)
    meas, weights = np.array(meas), np.array(weights)

    # p_s − p_t = s(s, t), con p_0 = 0
    A = np.zeros((len(pairs), T))
    for k, (a, b) in enumerate(pairs):
        A[k, a], A[k, b] = 1, -1
    A = A[:, 1:]
    huber = np.ones(len(pairs))
    for _ in range(10):
        w = np.sqrt(weights * huber)[:, None]
        P = np.linalg.lstsq(A * w, meas * w, rcond=None)[0]
        residual = np.hypot(*(A @ P - meas).T)
        huber = np.where(residual <= 1, 1, 1 / np.maximum(residual, 1e-9))
    return np.vstack([np.zeros(2), P]), residual


def shift_frame(frame: np.ndarray, valid: np.ndarray, shift: np.ndarray) -> np.ndarray:
    """Desplaza un frame; los píxeles que tocan "sin dato" quedan en 0 °C."""
    from scipy import ndimage

    out = ndimage.shift(np.where(valid, frame, 0).astype(np.float64), shift, order=1, cval=0)
    mask = ndimage.shift(valid.astype(np.float64), shift, order=1, cval=0)
    return np.where(mask > 0.999, out, 0).astype(np.float32)


def alinear(escenario, cool=50.0, upsample=20, rango=(1.0, 99.9)):
    escenario = Path(escenario)
    paths, stack = leer_tiff(escenario / "geo_thermal_tiff_celsius")
    valid = stack > 0
    r0, r1, c0, c1 = largest_rectangle(valid.all(0))
    print(f"{escenario.name}: {len(paths)} frames {stack.shape[1]}×{stack.shape[2]}, "
          f"medición en filas {r0}:{r1}, columnas {c0}:{c1}")

    positions, residual = measure_positions(stack[:, r0:r1, c0:c1], cool, upsample)
    correction = -positions
    print(f"  {len(residual)} pares, residuo mediana {np.median(residual):.2f} px, p95 {np.percentile(residual, 95):.2f} px")
    print(f"  corrección máxima: {np.hypot(*correction.T).max():.2f} px")

    tiff_dir = escenario / "geo_thermal_tiff_celsius_aligned"
    jpg_dir = escenario / "raw_thermal_jpg_aligned"
    tiff_dir.mkdir(exist_ok=True)
    jpg_dir.mkdir(exist_ok=True)
    aligned = np.stack([shift_frame(stack[t], valid[t], correction[t]) for t in range(len(stack))])
    vmin, vmax = np.percentile(aligned[aligned > 0], rango)
    for path, frame in zip(paths, aligned):
        with rasterio.open(path) as src:
            profile = src.profile
        with rasterio.open(tiff_dir / path.name, "w", **profile) as dst:
            dst.write(frame, 1)
        Image.fromarray(to_rgb(frame, vmin, vmax)).save(jpg_dir / path.name.replace("_ref_geo.TIFF", ".jpg"), quality=95)

    with open(escenario / "shifts.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["frame", "shift_row", "shift_col"])
        for path, c in zip(paths, correction):
            w.writerow([path.stem.replace("_ref_geo", ""), *np.round(c, 3)])
    print(f"  JPG con escala {vmin:.1f}–{vmax:.1f} °C → {jpg_dir}")

In [ ]:
for esc in ESCENARIOS:
    alinear(esc)

## 3. Recorte cuadrado

Toma el cuadrado más grande con dato en **todos** los frames alineados (programación dinámica, O(H·W); entre empates,
el más cercano al centro de la zona válida) y guarda los GeoTIFF recortados (con el transform ajustado, siguen
georreferenciados) y sus JPG.

In [ ]:
def largest_square(mask: np.ndarray) -> tuple:
    """Cuadrado más grande con todo True. Regresa (fila, col, lado) de la esquina superior izquierda."""
    H, W = mask.shape
    side = np.zeros((H + 1, W + 1), dtype=int)  # lado del cuadrado que termina en (i, j)
    for i in range(H):
        row, up = side[i + 1], side[i]
        m = mask[i]
        for j in range(W):
            if m[j]:
                row[j + 1] = 1 + min(up[j], up[j + 1], row[j])
    best = side.max()
    ys, xs = np.nonzero(side == best)
    cy, cx = np.argwhere(mask).mean(axis=0)
    # centro de cada candidato: (fin − lado/2)
    k = np.argmin((ys - best / 2 - cy) ** 2 + (xs - best / 2 - cx) ** 2)
    return ys[k] - best, xs[k] - best, best


def recortar(escenario, rango=(1.0, 99.9)):
    from rasterio.windows import Window

    escenario = Path(escenario)
    paths, stack = leer_tiff(escenario / "geo_thermal_tiff_celsius_aligned")
    r, c, n = largest_square((stack > 0).all(0))
    crop = stack[:, r:r + n, c:c + n]
    print(f"{escenario.name}: {len(paths)} frames {stack.shape[1]}×{stack.shape[2]}, "
          f"cuadrado {n}×{n} en filas {r}:{r + n}, columnas {c}:{c + n}")

    tiff_dir = escenario / f"geo_thermal_tiff_celsius_aligned_{n}"
    jpg_dir = escenario / f"raw_thermal_jpg_aligned_{n}"
    tiff_dir.mkdir(exist_ok=True)
    jpg_dir.mkdir(exist_ok=True)
    vmin, vmax = np.percentile(crop, rango)
    window = Window(c, r, n, n)
    for path, frame in zip(paths, crop):
        with rasterio.open(path) as src:
            profile = src.profile
            transform = src.window_transform(window)
        profile.update(width=n, height=n, transform=transform)
        with rasterio.open(tiff_dir / path.name, "w", **profile) as dst:
            dst.write(frame, 1)
        Image.fromarray(to_rgb(frame, vmin, vmax)).save(jpg_dir / path.name.replace("_ref_geo.TIFF", ".jpg"), quality=95)
    print(f"  JPG con escala {vmin:.1f}–{vmax:.1f} °C → {jpg_dir}")

In [ ]:
for esc in ESCENARIOS:
    recortar(esc)

## 4. Reducción a n×n

Reduce el cuadrado a `size` × `size` con un kernel separable: Y_t = K_r · X_t · K_cᵀ, donde cada fila de K convoluciona
un eje con el kernel y lo muestrea en `size` centros equiespaciados (paso s = lado/n). Con el kernel gaussiano
(σ = s/2, truncado en 3σ) cada fila suma 1, así que la temperatura media no cambia. Guarda los GeoTIFF n×n (transform
escalado) y sus JPG; con `zoom = k` guarda además los JPG ampliados k veces (cada celda un bloque k×k).

In [ ]:
KERNELS = ("box", "tent", "gaussian")


def kernel_matrix(L: int, N: int, kind: str) -> np.ndarray:
    """Matriz (N, L) que convoluciona un eje de longitud L con el kernel y lo muestrea en N centros. Cada fila suma 1."""
    s = L / N
    centers = (np.arange(N) + 0.5) * s - 0.5
    j = np.arange(L)[None, :]
    d = j - centers[:, None]
    if kind == "box":
        # promedio por área: traslape exacto de cada píxel con la ventana [c - s/2, c + s/2]
        w = np.minimum(j + 0.5, centers[:, None] + s / 2) - np.maximum(j - 0.5, centers[:, None] - s / 2)
        w = np.clip(w, 0, None)
    elif kind == "tent":
        w = np.clip(1 - np.abs(d) / s, 0, None)
    elif kind == "gaussian":
        sigma = s / 2
        w = np.exp(-d ** 2 / (2 * sigma ** 2)) * (np.abs(d) <= 3 * sigma)
    else:
        raise ValueError(kind)
    return w / w.sum(axis=1, keepdims=True)


def downsample(stack: np.ndarray, size: int, kind: str) -> np.ndarray:
    Kr = kernel_matrix(stack.shape[1], size, kind)
    Kc = kernel_matrix(stack.shape[2], size, kind)
    return (Kr @ stack.astype(np.float64) @ Kc.T).astype(np.float32)


def reducir(escenario, size=100, kernel="gaussian", rango=(1.0, 99.9), zoom=0):
    from rasterio.transform import Affine

    escenario = Path(escenario)
    # el cuadrado del recorte es el de mayor lado (las reducidas también siguen <nombre>_<n>)
    fuentes = [d for d in escenario.glob("geo_thermal_tiff_celsius_aligned_*") if d.name.rsplit("_", 1)[1].isdigit()]
    if not fuentes:
        sys.exit(f"No hay geo_thermal_tiff_celsius_aligned_<lado>/ en {escenario}")
    src_dir = max(fuentes, key=lambda d: int(d.name.rsplit("_", 1)[1]))
    paths, stack = leer_tiff(src_dir)
    side = stack.shape[1]
    if size >= side:
        sys.exit(f"size {size} no es menor que el lado de {src_dir.name} ({side})")
    small = downsample(stack, size, kernel)
    print(f"{escenario.name}: {len(paths)} frames {side}×{side} → {size}×{size} ({kernel}), {side / size:.2f} px por celda")

    tiff_dir = escenario / f"geo_thermal_tiff_celsius_aligned_{size}"
    jpg_dir = escenario / f"raw_thermal_jpg_aligned_{size}"
    zoom_dir = escenario / f"raw_thermal_jpg_aligned_{size}_x{zoom}"
    tiff_dir.mkdir(exist_ok=True)
    jpg_dir.mkdir(exist_ok=True)
    if zoom > 1:
        zoom_dir.mkdir(exist_ok=True)
    vmin, vmax = np.percentile(small, rango)
    for path, frame in zip(paths, small):
        with rasterio.open(path) as src:
            profile = src.profile
            transform = src.transform * Affine.scale(side / size)
        profile.update(width=size, height=size, transform=transform)
        for key in ("blockxsize", "blockysize", "tiled"):
            profile.pop(key, None)
        with rasterio.open(tiff_dir / path.name, "w", **profile) as dst:
            dst.write(frame, 1)
        name = path.name.replace("_ref_geo.TIFF", ".jpg")
        rgb = to_rgb(frame, vmin, vmax)
        Image.fromarray(rgb).save(jpg_dir / name, quality=95)
        if zoom > 1:
            Image.fromarray(rgb.repeat(zoom, 0).repeat(zoom, 1)).save(zoom_dir / name, quality=95)
    print(f"  JPG con escala {vmin:.1f}–{vmax:.1f} °C → {jpg_dir}")

In [ ]:
for esc in ESCENARIOS:
    reducir(esc, size=100)

## 5. Número de estados (análisis)

No escribe datos de entrenamiento; sirvió para elegir cómo dividir la temperatura.

**K-means y codo.** Junta las temperaturas de todas las celdas de todos los frames de un escenario (el estado i debe
ser el mismo rango de °C en toda la simulación), toma una muestra, ajusta *k-means* 1D para k = `k_min` … `k_max` y
elige k por el método del codo (Kneedle: el punto de la curva de inercia normalizada más lejano a la recta entre sus
extremos); la silueta solo se reporta. Guarda `clusters_<n>.png` (codo y silueta) y `clusters_<n>.json` (curva, k,
centroides y umbrales); con `k` fijo, `clusters_<n>_k<k>.*`. En los tres escenarios el codo sale en k = 4 y el
estado 0 abarca del 66 al 99 % de las celdas.

**Comparación visual.** Escribe los estados en paleta turbo (`raw_thermal_jpg_aligned_<n>_x<zoom>_<método><K>/`, más
`_leyenda.png` con el rango de °C de cada estado) con cuatro divisiones: `kmeans` (umbrales del codo con `k` fijo,
propios de cada escenario) y, con el mínimo y el máximo de los escenarios dados, `ancho` (mismo ancho en °C),
`cuantil` (mismo número de celdas) y `log` (mismo ancho en log °C). Se eligió `log` con 15 estados.

In [ ]:
# estilo de notebooks/gol_proportions.nb
FRAME, GRID = "#666666", "#d9d9d9"
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"


def elbow_k(ks: list, inertias: list) -> int:
    """Kneedle: k con la mayor distancia a la recta que une los extremos de la curva normalizada."""
    x = (np.array(ks) - ks[0]) / (ks[-1] - ks[0])
    y = (np.array(inertias) - inertias[-1]) / (inertias[0] - inertias[-1])
    # la curva de inercia es decreciente y convexa: la recta va de (0,1) a (1,0)
    return ks[int(np.argmax((1 - x) - y))]


def scan_k(sample: np.ndarray, k_min: int, k_max: int, seed: int, sil_size: int) -> dict:
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score

    ks, inertias, silhouettes = [], [], []
    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, n_init=4, random_state=seed).fit(sample)
        sil = silhouette_score(sample, km.labels_, sample_size=min(sil_size, len(sample)), random_state=seed)
        ks.append(k)
        inertias.append(float(km.inertia_))
        silhouettes.append(float(sil))
        print(f"  k={k:2d}  inercia={km.inertia_:.4g}  silueta={sil:.4f}")
    return {"k": ks, "inertia": inertias, "silhouette": silhouettes}


def plot_elbow(path: Path, scan: dict, k: int, title: str, label: str):
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "axes.edgecolor": FRAME, "axes.labelcolor": "black", "xtick.color": FRAME, "ytick.color": FRAME,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "font.size": 10,
        "xtick.direction": "in", "ytick.direction": "in", "font.family": "serif",
    })
    ks, inertia, sil = np.array(scan["k"]), np.array(scan["inertia"]), np.array(scan["silhouette"])
    i = int(np.where(ks == k)[0][0])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3), dpi=200)
    ax1.plot(ks, inertia, color=BLUE, lw=2, marker="o", ms=4)
    ax1.plot(ks[[0, -1]], inertia[[0, -1]], color=FRAME, lw=1, ls=":")     # el codo es el punto más lejano a ella
    ax1.plot(k, inertia[i], "o", ms=10, mfc="none", mec=ORANGE, mew=2)
    ax1.annotate(f"k = {k}", (k, inertia[i]), xytext=(-12 if k == ks[-1] else 12, 10), textcoords="offset points",
                 color=ORANGE, fontweight="bold", ha="right" if k == ks[-1] else "left")
    ax1.set(xlabel="Número de clusters k", ylabel="Inercia (°C²)", xticks=ks, title=label)
    ax1.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax2.plot(ks, sil, color=GREEN, lw=2, marker="o", ms=4)
    ax2.plot(k, sil[i], "o", ms=10, mfc="none", mec=ORANGE, mew=2)
    ax2.set(xlabel="Número de clusters k", ylabel="Coeficiente de silueta", xticks=ks, title="Silueta")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path)
    plt.show()


def kmeans_estados(escenario, size=100, k_min=2, k_max=10, sample=200_000, sil_size=10_000, seed=0, k=None):
    from sklearn.cluster import KMeans

    escenario = Path(escenario)
    paths, temps = leer_tiff(escenario / f"geo_thermal_tiff_celsius_aligned_{size}")
    values = temps.astype(np.float64).reshape(-1, 1)
    rng = np.random.default_rng(seed)
    muestra = values[rng.choice(len(values), size=min(sample, len(values)), replace=False)]
    print(f"{escenario.name}: {len(paths)} frames {size}×{size} = {len(values):,} valores, muestra {len(muestra):,}")

    if k is not None:
        k_max = max(k_max, k)
    scan = scan_k(muestra, k_min, k_max, seed, sil_size)
    elbow = elbow_k(scan["k"], scan["inertia"])
    k_usado = k if k is not None else elbow
    print(f"  codo: k = {elbow}  (silueta máxima en k = {scan['k'][int(np.argmax(scan['silhouette']))]})"
          + (f"; se usa k = {k}" if k is not None else ""))

    km = KMeans(n_clusters=k_usado, n_init=10, random_state=seed).fit(muestra)
    centroids = np.sort(km.cluster_centers_.ravel())
    thresholds = (centroids[:-1] + centroids[1:]) / 2
    counts = np.bincount(np.digitize(values.ravel(), thresholds), minlength=k_usado)
    print("  centroides °C: " + ", ".join(f"{c:.1f}" for c in centroids))
    print("  umbrales °C:   " + ", ".join(f"{t:.1f}" for t in thresholds))

    stem = f"clusters_{size}" + (f"_k{k}" if k is not None else "")
    plot_elbow(escenario / f"{stem}.png", scan, k_usado, f"{escenario.name} ({size}×{size}, {len(paths)} frames)",
               "Método del codo" if k is None else f"k fijado (codo en k = {elbow})")
    with open(escenario / f"{stem}.json", "w") as f:
        json.dump({"scan": scan, "k": k_usado, "elbow_k": elbow, "centroids": centroids.round(3).tolist(),
                   "thresholds": thresholds.round(3).tolist(), "fraction": (counts / counts.sum()).round(5).tolist(),
                   "sample": len(muestra), "seed": seed}, f, indent=2)


def leyenda(path: Path, thresholds: np.ndarray, colors: np.ndarray, lo: float, hi: float, title: str):
    import matplotlib.pyplot as plt

    k = len(colors)
    edges = np.concatenate([[lo], thresholds, [hi]])
    fig, ax = plt.subplots(figsize=(3.6, 0.32 * k + 0.6), dpi=150)
    for s in range(k):
        y = k - 1 - s
        ax.add_patch(plt.Rectangle((0, y + 0.1), 0.8, 0.8, color=colors[s] / 255))
        ax.text(1.0, y + 0.5, f"{s}:  {edges[s]:.1f} – {edges[s + 1]:.1f} °C", va="center", family="serif")
    ax.set(xlim=(0, 5), ylim=(0, k), title=title)
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def comparar_divisiones(escenarios, k=10, size=100, zoom=10, metodos=("kmeans", "ancho", "cuantil", "log")):
    data = {Path(sc): leer_tiff(Path(sc) / f"geo_thermal_tiff_celsius_aligned_{size}") for sc in escenarios}
    every = np.concatenate([t.ravel() for _, t in data.values()])
    lo, hi = float(every.min()), float(every.max())
    common = {"ancho": np.linspace(lo, hi, k + 1)[1:-1],
              "cuantil": np.percentile(every, np.linspace(0, 100, k + 1)[1:-1]),
              "log": np.geomspace(lo, hi, k + 1)[1:-1]}
    print(f"rango común: {lo:.1f} – {hi:.1f} °C")
    for sc, (paths, temps) in data.items():
        for metodo in metodos:
            if metodo == "kmeans":
                nombre = sc / f"clusters_{size}_k{k}.json"
                if not nombre.exists():
                    print(f"{sc.name} kmeans: falta {nombre.name} (correr kmeans_estados con k={k}), se omite")
                    continue
                thresholds = np.array(json.load(open(nombre))["thresholds"])
            else:
                thresholds = common[metodo]
            colors = palette(len(thresholds) + 1)
            states = np.digitize(temps, thresholds)
            out = sc / f"raw_thermal_jpg_aligned_{size}_x{zoom}_{metodo}{k}"
            out.mkdir(exist_ok=True)
            for path, st in zip(paths, states):
                Image.fromarray(colors[st].repeat(zoom, 0).repeat(zoom, 1)).save(
                    out / path.name.replace("_ref_geo.TIFF", ".jpg"), quality=95)
            leyenda(out / "_leyenda.png", thresholds, colors, float(temps.min()), float(temps.max()), f"{sc.name} · {metodo}{k}")
            frac = np.bincount(states.ravel(), minlength=len(colors)) / states.size * 100
            print(f"{sc.name} {metodo:7s} cortes °C: " + " ".join(f"{t:.1f}" for t in thresholds))
            print(f"{'':{len(sc.name)}} {'':7s} % por estado: " + " ".join(f"{f:.1f}" for f in frac))

In [ ]:
for esc in ESCENARIOS:
    kmeans_estados(esc)
kmeans_estados(ESCENARIOS[2], k_max=15, k=15)
comparar_divisiones(ESCENARIOS[2:], k=15)

## 6. Estados logarítmicos

Define `k` estados con cortes iguales en log(°C) entre el mínimo y el máximo de **todos** los escenarios dados, así el
estado s es el mismo rango de °C en todos ellos, y asigna a cada celda su estado (0 = más frío). Guarda por escenario,
con la misma estructura que el resto de `CA/`, `evo.npy` / `evo.mat` (int32 (T, n, n)), `evo_celsius.npy` (°C antes de
cortar), `evo.json` (metadatos, rango de cada estado, Δt), `evo.gif` y `evo_final.png`, y escribe la paleta de la GUI
`GUI/colors/flame_log<k>.csv` (turbo en k pasos). Con 15 estados en la selección 2: 25.3–604.6 °C.

In [ ]:
EXIF_DATETIME = 306


def load_timestamps(paths: list, jpg_dir: Path) -> list:
    """Los GeoTIFF no traen fecha; se toma del JPG original con el mismo número (IRX_XXXX)."""
    stamps = []
    for p in paths:
        jpg = jpg_dir / (p.name.split("_ref")[0] + ".jpg")
        raw = Image.open(jpg).getexif().get(EXIF_DATETIME) if jpg.exists() else None
        stamps.append(datetime.strptime(raw, "%Y:%m:%d %H:%M:%S") if raw else None)
    return stamps


def estados_log(escenarios, k=15, size=100, output=None, step=0, zoom=4, fps=10):
    """output: carpeta de salida (default: CA/FLAME3/<selección>, la carpeta que contiene al escenario)."""
    escenarios = [Path(s) for s in escenarios]
    data = {sc: leer_tiff(sc / f"geo_thermal_tiff_celsius_aligned_{size}") for sc in escenarios}
    lo = float(min(t.min() for _, t in data.values()))
    hi = float(max(t.max() for _, t in data.values()))
    edges = np.geomspace(lo, hi, k + 1)
    thresholds = edges[1:-1]
    print(f"{k} estados log entre {lo:.2f} y {hi:.2f} °C (factor {edges[1] / edges[0]:.3f})")
    print("  cortes °C: " + ", ".join(f"{t:.1f}" for t in thresholds))

    colors = palette(k)
    paleta = f"GUI/colors/flame_log{k}.csv"
    (SIMULATOR / paleta).write_text("".join("#%02X%02X%02X\n" % tuple(c) for c in colors))
    print(f"  paleta GUI → {paleta}")
    for sc, (paths, temps) in data.items():
        states = np.digitize(temps, thresholds).astype(np.int32)
        stamps = load_timestamps(paths, sc / "raw_thermal_jpg")
        dt = np.diff([s.timestamp() for s in stamps if s])
        cuenta = np.bincount(states.ravel(), minlength=k)
        out = (Path(output) if output else CA / sc.resolve().parent.name) / sc.name
        guardar_evolucion(out, states, temps, colors, {
            "simulation": "flame3",
            "source": str((sc / f"geo_thermal_tiff_celsius_aligned_{size}").resolve()),
            "files": [p.name for p in paths],
            "timestamps": [s.isoformat() if s else None for s in stamps],
            "shape": list(states.shape),
            "size": size,
            "pipeline": ["alinear", "recortar", "reducir (gaussian)", "estados_log"],
            "k": k,
            "k_selection": "log",
            "range_celsius": [lo, hi],
            "range_scenarios": [s.name for s in escenarios],
            "edges_celsius": edges.tolist(),
            "thresholds_celsius": thresholds.tolist(),
            "state_fraction": (cuenta / states.size).tolist(),
            "state_count": cuenta.tolist(),
            "states": [f"{s}: {edges[s]:.1f} – {edges[s + 1]:.1f} °C" for s in range(k)],
            "palette": paleta,
            "time_step_seconds": float(np.median(dt)),
        }, zoom, fps)
        gaps = [(i, round(float(d))) for i, d in enumerate(dt) if d > 10]
        print(f"{sc.name}: {states.shape} int32, estados {states.min()}–{states.max()}, "
              f"Δt mediana {np.median(dt):.0f} s, huecos > 10 s: {gaps if gaps else 'ninguno'} → {out}")
        if step:
            secuencias(states, out, step)

In [ ]:
estados_log(ESCENARIOS, k=15, step=6)

## 7. Unión de estados fríos

Los estados de `merge` (consecutivos) pasan a ser uno y los demás se recorren hacia abajo. Escribe en
`<escenario>/<name>/` la misma estructura (cortes y rangos actualizados) y la paleta `GUI/colors/flame_log<K>.csv`: el
estado unido toma el color del de en medio de los que une y los demás conservan el suyo, así el mismo rango de °C
tiene el mismo color. `estados13` une 0–2; `estados12` une 0–3 (25.3–59.0 °C), que es la que se usa.

In [ ]:
def unir_estados(escenarios, merge, name, step=0, zoom=4, fps=10):
    unir = sorted(merge)
    assert unir == list(range(unir[0], unir[-1] + 1)), "los estados a unir deben ser consecutivos"
    primero, n_quita = unir[0], len(unir) - 1
    for esc in map(Path, escenarios):
        meta = json.loads((esc / "evo.json").read_text())
        k = meta["k"]
        mapa = np.array([s if s <= primero else max(primero, s - n_quita) for s in range(k)], dtype=np.int32)
        k_nuevo = k - n_quita
        bordes = meta["edges_celsius"]
        bordes_nuevos = bordes[:primero + 1] + bordes[unir[-1] + 1:]
        evo = mapa[np.load(esc / "evo.npy")]

        viejos = leer_paleta(meta["palette"])
        colores = [viejos[s] for s in range(primero)] + [viejos[unir[len(unir) // 2]]] + viejos[unir[-1] + 1:]
        paleta = f"GUI/colors/flame_log{k_nuevo}.csv"
        (SIMULATOR / paleta).write_text("\n".join(colores) + "\n")

        cuenta = np.bincount(evo.ravel(), minlength=k_nuevo)
        meta.update({
            "k": k_nuevo,
            "merged_states": {"from": unir, "into": primero, "source_k": k},
            "edges_celsius": bordes_nuevos,
            "thresholds_celsius": bordes_nuevos[1:-1],
            "states": [f"{s}: {bordes_nuevos[s]:.1f} – {bordes_nuevos[s + 1]:.1f} °C" for s in range(k_nuevo)],
            "state_count": cuenta.tolist(),
            "state_fraction": (cuenta / evo.size).tolist(),
            "palette": paleta,
            "pipeline": meta["pipeline"] + [f"unir_estados {' '.join(map(str, unir))}"],
        })
        out = esc / name
        guardar_evolucion(out, evo, np.load(esc / "evo_celsius.npy"), hex_a_rgb(colores), meta, zoom, fps)
        print(f"{esc.name}: estados {unir} → {primero}; {k} → {k_nuevo} estados, {evo.shape} → {out}")
        if step:
            secuencias(evo, out, step)

In [ ]:
SEL2 = [CA / "seleccion2" / f"flame_plot_{i}" for i in (1, 2, 3)]
unir_estados(SEL2, [0, 1, 2], "estados13", step=6)
unir_estados(SEL2, [0, 1, 2, 3], "estados12", step=6)

## 8. Estados de quemado

Agrega B estados de quemado a una evolución de K estados de temperatura (K + B estados). Regla por celda, frame a
frame (t = estado de temperatura 0…K−1):

- Si la temperatura baja (t actual < t anterior), la celda queda quemada.
- Si sube, deja de estar quemada. Con `rise` = R solo deja de estarlo si su temperatura en °C supera en más de R °C la
  mínima que alcanzó desde que se quemó; así las subidas pequeñas (ruido) no la regresan a sin quemar.
- Si no, conserva lo que tenía.

Sin quemar el estado es t; quemada es K + ⌊t·B/K⌋ (con B = K/2 las temperaturas se unen de dos en dos). El primer frame
es todo sin quemar. Con `until` = `IRX_NNNN` se queda con los frames hasta ese (como la regla solo mira hacia atrás,
cortar antes o después da lo mismo). La paleta `GUI/colors/flame_log<K>_quemado<B>.csv` conserva los K colores y los
quemados van de negro (el más frío) a blanco. Las secuencias se toman después de calcular los estados con todos los
frames. Con `jpg` = Z escribe cada frame en `jpg_x<Z>/` (celdas de Z×Z px, para elegir recortes).

Datos del experimento final (`BurnedStates`, 12 + 6 estados, umbral de 30 °C, paso de 30 s): `flame_plot_1` = plot 1
`IRX_0662`–`0777` y `flame_plot_2` = plot 2 `IRX_1189`–`1254` (en ambos cortes el fuego activo ya es < 0.15 %).

In [ ]:
def quemados(temp, k, b, celsius=None, subida=0):
    """Estados 0…k+b−1 a partir de los estados de temperatura 0…k−1 (frames, filas, columnas)."""
    grupo = k + np.arange(k) * b // k
    evo = np.empty_like(temp)
    evo[0] = temp[0]
    quemado = np.zeros(temp.shape[1:], dtype=bool)
    minimo = np.zeros(temp.shape[1:], dtype=np.float32)
    for f in range(1, len(temp)):
        baja = temp[f] < temp[f - 1]
        sube = celsius[f] - minimo > subida if subida > 0 else temp[f] > temp[f - 1]
        nuevo = np.where(quemado & sube, False, np.where(baja, True, quemado))
        if subida > 0:
            minimo = np.where(nuevo & ~quemado, celsius[f], np.minimum(minimo, celsius[f]))
        quemado = nuevo
        evo[f] = np.where(quemado, grupo[temp[f]], temp[f])
    return evo


def estados_quemado(origen, out, until=None, burned=0, rise=0, step=0, jpg=0, zoom=4, fps=10):
    origen, out = Path(origen), Path(out)
    meta = json.loads((origen / "evo.json").read_text())
    k = meta["k"]
    b = burned or k
    assert 1 <= b <= k, "burned debe estar entre 1 y K"
    temp = np.load(origen / "evo.npy")
    celsius = np.load(origen / "evo_celsius.npy")
    if until:
        n = [f.split("_ref")[0] for f in meta["files"]].index(until) + 1
        temp, celsius = temp[:n], celsius[:n]
        meta.update(files=meta["files"][:n], timestamps=meta["timestamps"][:n], shape=list(temp.shape))
    evo = quemados(temp, k, b, celsius, rise)

    grises = np.linspace(0, 255, b).round().astype(int)
    colores = leer_paleta(meta["palette"]) + [f"#{g:02X}{g:02X}{g:02X}" for g in grises]
    paleta = f"GUI/colors/{Path(meta['palette']).stem}_quemado{b}.csv"
    (SIMULATOR / paleta).write_text("\n".join(colores) + "\n")
    rgb = hex_a_rgb(colores)

    bordes = meta["edges_celsius"]
    temps = [[t for t in range(k) if t * b // k == g] for g in range(b)]
    cuenta = np.bincount(evo.ravel(), minlength=k + b)
    meta.update({
        "k": k + b,
        "burned_states": {"temperature_k": k, "burned_k": b, "groups": temps, "rise_celsius": rise,
                          "rule": "baja → quemado; sube (más de rise_celsius °C sobre el mínimo desde que se"
                                  " quemó, si rise_celsius > 0) → sin quemar; si no → conserva"},
        "states": [f"{s}: {bordes[s]:.1f} – {bordes[s + 1]:.1f} °C" for s in range(k)]
                  + [f"{k + g}: {bordes[ts[0]]:.1f} – {bordes[ts[-1] + 1]:.1f} °C (quemado, temperaturas {ts[0]}–{ts[-1]})"
                     for g, ts in enumerate(temps)],
        "state_count": cuenta.tolist(),
        "state_fraction": (cuenta / evo.size).tolist(),
        "palette": paleta,
        "pipeline": meta["pipeline"] + [f"estados_quemado --burned {b} --rise {rise:g}" + (f" --until {until}" if until else "")],
    })
    guardar_evolucion(out, evo, celsius, rgb, meta, zoom, fps)
    if jpg:
        (out / f"jpg_x{jpg}").mkdir(exist_ok=True)
        for nombre, st in zip(meta["files"], evo):
            Image.fromarray(rgb[st].repeat(jpg, 0).repeat(jpg, 1)).save(
                out / f"jpg_x{jpg}" / nombre.replace("_ref_geo.TIFF", ".jpg"), quality=95)
    print(f"{origen}: {k} → {k + b} estados ({b} quemados), {evo.shape} → {out}")
    if step:
        secuencias(evo, out, step)

In [ ]:
BURNED = CA / "seleccion2" / "BurnedStates"
estados_quemado(SEL2[1] / "estados12", BURNED / "flame_plot_1", until="IRX_0777", burned=6, rise=30, step=6)
estados_quemado(SEL2[2] / "estados12", BURNED / "flame_plot_2", until="IRX_1254", burned=6, rise=30, step=6)

## Anexo: quitar la ignición

En FLAME 3 el fuego lo prende una persona trazando líneas, y esa parte no es dinámica del fuego. En la selección 1
(experimentos anteriores, `CA/FLAME3/seleccion1/sin_ignicion/`) se quitaron los primeros frames de cada escenario ya
convertido a estados, con la misma discretización; en la selección 2 la ignición se deja fuera al elegir los frames.

In [ ]:
def quitar_ignicion(origen, salida, inicio: dict, zoom=4, fps=10):
    """inicio: {escenario: primer frame que se conserva (índice desde 0)}."""
    for name, start in inicio.items():
        src = Path(origen) / name
        states = np.load(src / "evo.npy")[start:]
        temps = np.load(src / "evo_celsius.npy")[start:]
        meta = json.loads((src / "evo.json").read_text())
        k = meta["k"]
        meta.update({
            "files": meta["files"][start:],
            "timestamps": meta["timestamps"][start:],
            "shape": list(states.shape),
            "first_frame": start,
            "trimmed_from": str(src.resolve()),
            "state_fraction": (np.bincount(states.ravel(), minlength=k) / states.size).tolist(),
            "state_count": np.bincount(states.ravel(), minlength=k).tolist(),
        })
        meta["pipeline"] = meta["pipeline"] + ["quitar_ignicion"]
        out = Path(salida) / name
        guardar_evolucion(out, states, temps, hex_a_rgb(leer_paleta(meta["palette"])), meta, zoom, fps)
        print(f"{name}: frames {start}–{start + len(states) - 1} ({meta['files'][0]} – {meta['files'][-1]}), "
              f"{states.shape} → {out}")

In [ ]:
SEL1 = CA / "seleccion1"
quitar_ignicion(SEL1, SEL1 / "sin_ignicion", {"flame_plot_1": 133, "flame_plot_2": 171, "flame_plot_3": 19})